# Greediness Metrics for Bandits

Using this notebook to code up the three metrics from "LLMs are Greedy Agents", will shift to a source Python file when done.

In [2]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any
from pprint import pprint as pp

In [3]:
styles = [3, 5, 6]
efforts = ['default']#,"medium","high"]
# styles = [style + '_' + effort for effort in efforts for style in styles]
ucb_path: str = f'../src/optimal_explorer/strategies/bandits/logs/ucb_mab_bernoulli.jsonl'
random_path: str = f'../src/optimal_explorer/strategies/bandits/logs/Random_bernoulli.jsonl'


models = [
    "Gemini Pro 2.5",
    "DeepSeek R1",
    # "Claude Opus 4",
    # "Claude 3.5 Sonnet",
    # "OpenAI o3",
]

model_ids = [
    "google/gemini-2.5-pro",
    "deepseek/deepseek-r1-0528",
    # "anthropic/claude-opus-4",
    # "anthropic/claude-3.5-sonnet",
    # "openai/o3",
]

results_paths: List[str] = [
    f'../src/optimal_explorer/strategies/bandits/logs/game_results/style{style}_{model_id.split("/")[-1]}_default_bernoulli.jsonl'
    for style in styles for model_id in model_ids
]

In [11]:
with open(ucb_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        pp(data)
        break

{'cumulative_regret': 12.655015386153433,
 'env_config': {'arm_names': ['1',
                              '2',
                              '3',
                              '4',
                              '5',
                              '6',
                              '7',
                              '8',
                              '9',
                              '10'],
                'max_steps': 50,
                'noise_level': 'medium',
                'num_arms': 10,
                'reward_type': 'bernoulli'},
 'game_id': 0,
 'history': [{'action': '1',
              'attempt': 1,
              'regret': 0.41484925657370453,
              'reward': 0},
             {'action': '2',
              'attempt': 2,
              'regret': 0.2484733941286098,
              'reward': 1},
             {'action': '3',
              'attempt': 3,
              'regret': 0.3608993844293854,
              'reward': 1},
             {'action': '4',
              'attempt'

In [12]:
results = []

for results_path in results_paths:
    style = results_path.split('_')[2][-1]
    print(style)
    model_id = results_path.split('_')[2]
    with open(results_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            model = data['model']
            regret = data['regret_per_attempt']
            results.append({
                'game_id': data['game_id'],
                'model': model + f' (s={style})',
                'regret': regret,
                'length': len(data['history']),
                'style': style,
                'history': data['history'],
            })
results_df = pd.DataFrame(results)

3
3
5
5
6
6


In [13]:
ucb = []
random = []

with open(ucb_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        # regret = [sum(data['regret_per_attempt'][:i]) for i in range(data['num_attempts'])]
        regret = data['regret_per_attempt']
        ucb.append({
            'game_id': data['game_id'],
            'model': 'UCB',
            'regret': regret,
            'length': data['num_attempts'],
            'history': data['history'],
        })
ucb_df = pd.DataFrame(ucb)

with open(random_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        # regret = [sum(data['regret_per_attempt'][:i]) for i in range(data['num_attempts'])]
        regret = data['regret_per_attempt']
        random.append({
            'game_id': data['game_id'],
            'model': 'Random',
            'regret': regret,
            'length': data['num_attempts'],
            'history': data['history'],
        })
random_df = pd.DataFrame(random)

In [14]:
results_df = results_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
ucb_df = ucb_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
random_df = random_df.drop_duplicates(subset=['game_id', 'model'], keep='last')

In [15]:
random_df.head(5)

,game_id,model,regret,length,history
0,0,Random,"[0.0, 0.41484925657370453, 0.5802212416752516,...",50,"[{'attempt': 1, 'action': '9', 'reward': 1, 'r..."
1,1,Random,"[0.7202101186248132, 0.7202101186248132, 0.534...",50,"[{'attempt': 1, 'action': '3', 'reward': 0, 'r..."
2,2,Random,"[0.18394857373238682, 0.4146223323128212, 0.28...",50,"[{'attempt': 1, 'action': '4', 'reward': 0, 'r..."
3,3,Random,"[0.7707077784696018, 0.003346134585783367, 0.0...",50,"[{'attempt': 1, 'action': '7', 'reward': 1, 'r..."
4,4,Random,"[0.7601849591958654, 0.003590094811357414, 0.0...",50,"[{'attempt': 1, 'action': '6', 'reward': 0, 'r..."


In [28]:
results_df.columns

Index(['game_id', 'model', 'regret', 'length', 'style', 'history'], dtype='object')

In [ ]:
We define action coverage 𝐶𝑡 at step 𝑡 as the fraction of available actions that have been selected at least once, 𝐶𝑡 = {𝑎∈A: 𝑁𝑡 (𝑎)>0} | A | with 𝑁𝑡 (𝑎) representing the number of times action 𝑎 ∈ A has been selected until 𝑡.

In [27]:
results_df.iloc[90]['history']

[{'attempt': 1,
  'action': '1',
  'reward': 0,
  'regret': 0.7047419040659821,
  'belief': None},
 {'attempt': 2,
  'action': '2',
  'reward': 1,
  'regret': 0.5032766104529166,
  'belief': None},
 {'attempt': 3,
  'action': '3',
  'reward': 1,
  'regret': 0.5423112285071713,
  'belief': None},
 {'attempt': 4,
  'action': '1',
  'reward': 0,
  'regret': 0.7047419040659821,
  'belief': None},
 {'attempt': 5,
  'action': '4',
  'reward': 1,
  'regret': 0.6929523406168122,
  'belief': None},
 {'attempt': 6,
  'action': '5',
  'reward': 0,
  'regret': 0.0828563431567626,
  'belief': None},
 {'attempt': 7,
  'action': '6',
  'reward': 0,
  'regret': 0.7682121043418071,
  'belief': None},
 {'attempt': 8,
  'action': '7',
  'reward': 0,
  'regret': 0.9335906060736681,
  'belief': None},
 {'attempt': 9, 'action': '8', 'reward': 1, 'regret': 0.0, 'belief': None},
 {'attempt': 10,
  'action': '9',
  'reward': 1,
  'regret': 0.2728012267726728,
  'belief': None},
 {'attempt': 11,
  'action': '1'

In [29]:
from collections import defaultdict

def compute_action_coverage(df, action_space=10):
    """
    Computes action coverage C_t at each timestep for each model.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing a 'model' column and a 'history' column.
        action_space (int): The total number of possible actions (default is 10).
    
    Returns:
        dict: Keys are model names, values are lists of C_t values over time.
    """
    coverage_by_model = defaultdict(list)
    
    for _, row in df.iterrows():
        model = row['model']
        history = row['history']
        
        seen_actions = set()
        coverages = []
        
        for step in history:
            seen_actions.add(int(step['action']))
            C_t = len(seen_actions) / action_space
            coverages.append(C_t)
        
        coverage_by_model[model].append(coverages)
    
    # Optionally, average over games per model if there are multiple entries
    averaged_coverage_by_model = {}
    for model, coverage_lists in coverage_by_model.items():
        # Transpose list of lists to average across games at each timestep
        max_length = max(len(lst) for lst in coverage_lists)
        # Pad shorter lists with last value (assumes coverage remains the same if game ends early)
        padded = [lst + [lst[-1]] * (max_length - len(lst)) for lst in coverage_lists]
        # Compute mean across games
        avg_coverage = [sum(step_vals) / len(step_vals) for step_vals in zip(*padded)]
        averaged_coverage_by_model[model] = avg_coverage

    return averaged_coverage_by_model

In [30]:
coverage_dict = compute_action_coverage(results_df)

In [31]:
coverage_dict

{'google/gemini-2.5-pro (s=3)': [0.09999999999999996,
  0.19999999999999993,
  0.30000000000000027,
  0.3979999999999999,
  0.498,
  0.5980000000000005,
  0.6979999999999998,
  0.7959999999999998,
  0.8919999999999991,
  0.8959999999999992,
  0.8959999999999992,
  0.8979999999999992,
  0.8979999999999992,
  0.8979999999999992,
  0.8979999999999992,
  0.8979999999999992,
  0.8979999999999992,
  0.8979999999999992,
  0.8979999999999992,
  0.8979999999999992],
 'deepseek/deepseek-r1-0528 (s=3)': [0.09999999999999996,
  0.19999999999999993,
  0.30000000000000027,
  0.39599999999999996,
  0.49599999999999994,
  0.5940000000000005,
  0.6939999999999997,
  0.794,
  0.8939999999999992,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991,
  0.8999999999999991],
 'google/gemini-2.5-pro (s=5)': [0.09999999999999996,
  0.199999999999

In [41]:
import plotly.graph_objs as go
import plotly.express as px

def plot_action_coverage(coverage_dict):
    """
    Plots action coverage over time for each model using Plotly.

    Parameters:
        coverage_dict (dict): Output from `compute_action_coverage` function,
                              keys are model names and values are lists of C_t.
    """
    fig = go.Figure()

    color_map = px.colors.qualitative.Plotly  # Distinct colors for each model
    for i, (model, coverage_list) in enumerate(coverage_dict.items()):
        steps = list(range(1, len(coverage_list) + 1))
        fig.add_trace(go.Scatter(
            x=steps,
            y=coverage_list,
            mode='lines+markers',
            name=model,
            line=dict(color=color_map[i % len(color_map)], width=3),
            marker=dict(size=4)
        ))

    fig.update_layout(
        title="Action Coverage Over Steps",
        xaxis_title="Step",
        yaxis_title="Action Coverage Cₜ",
        yaxis=dict(range=[0, 1.05]),
        legend_title="Model",
        template="plotly_white",
        width=800,
        height=500
    )

    return fig

In [42]:
fig = plot_action_coverage(coverage_dict)

In [43]:
fig.show()